# 🪂 Lakeflow Designer (no-code ETL)

**Lakeflow Designer** is Databricks' no-code, node-based pipeline builder. Same engine as the SQL/Python pipelines you might write by hand — but you compose it by clicking + AI-assisted nodes, with a **live data preview at every step**.

We'll use it to build a small pipeline on top of the F1 silver tables you created in notebook 03.

> ⚠️ **Heads up — public preview.** Lakeflow Designer is rolling out gradually. If you don't see **Designer (no-code)** as an option when you go to create a pipeline, your workspace doesn't have it enabled yet — ask an admin to flip the preview on, or skip this notebook and come back later. The rest of the workshop still works without it.


## 🎯 What you'll build

A 3-node pipeline:

```text
  ┌────────────────────────────────────┐
  │ Source                             │
  │ main.default.f1_silver_race_results│
  └─────────────┬──────────────────────┘
                │  filter: finish_position <= 10
                ▼
  ┌────────────────────────────────────┐
  │ Transform                          │
  │ group by team                      │
  │ sum(points_earned) as season_points│
  │ count(*) as points_finishes        │
  └─────────────┬──────────────────────┘
                │
                ▼
  ┌────────────────────────────────────┐
  │ Sink                               │
  │ main.default.f1_team_points_finishes│
  └────────────────────────────────────┘
```

You'll see each row of data update live as you click through the nodes.

## 🪄 Fast path: let Genie Code build the pipeline

If you'd rather skip the clicking, paste this prompt into the **Genie Code** side panel inside Lakeflow Designer (✨ icon) and it will build the whole 3-node pipeline for you:

```
Build a 3-node Lakeflow Designer pipeline that does the following:

1) Source node — read from the Unity Catalog table
   `main.default.f1_silver_race_results`.

2) Filter node — keep only rows where `finish_position <= 10`.

3) Aggregate node — group by `team`, with:
     - sum(points_earned) AS season_points
     - count(*) AS points_finishes

4) Sink node — save the result as a Materialized view at
   `main.default.f1_team_points_finishes`.

Then run the pipeline and confirm it succeeds.
```

Want to learn the UI instead? The manual walkthrough is below — feel free to click through one or two transformations and then drop the prompt in to finish the rest.


## 🛠 Steps

### 1️⃣ Open Lakeflow Designer

1. In the left sidebar, go to **Lakeflow → Pipelines**.
2. Click **➕ Create pipeline → Designer (no-code)**.
3. Name it `F1 Team Points Finishes` and pick the same catalog/schema you used in notebook 03 (default: `main.default`).

![Create pipeline using Lakeflow Designer](./Images/Lakeflow_Designer_Create.png "Lakeflow Designer - Create")

### 2️⃣ Add the source node

1. Click **➕ Add source → Unity Catalog table**.
2. Pick `main.default.f1_silver_race_results`.
3. Click the node — the right-hand panel shows a **live preview** of the data. Confirm you see `race`, `finish_position`, `driver`, `team`, `points_earned`, etc.

### 3️⃣ Add a filter

1. From the source node, click **➕ → Filter**.
2. Set the expression: `finish_position <= 10`
3. Preview should drop the DNF/back-of-grid rows.

### 4️⃣ Add an aggregation

1. From the filter node, click **➕ → Aggregate**.
2. **Group by:** `team`
3. **Aggregations:**
   * `sum(points_earned)` → alias `season_points`
   * `count(*)` → alias `points_finishes`
4. Click the node — preview should show one row per team.

### 5️⃣ Add the sink

1. From the aggregate node, click **➕ → Save to Unity Catalog**.
2. Catalog: `main`, Schema: `default`, Table: `f1_team_points_finishes`.
3. Mode: **Materialized view** (so it refreshes when the upstream tables change).

### 6️⃣ Run it

1. Click **▶️ Run** in the top right.
2. Watch the nodes light up green as each step finishes.
3. When it's done, click **Schedule → every hour** (or whatever cadence you want). That's the 2-click scheduling Holly Smith mentioned.

![Completed 3-node Lakeflow Designer pipeline](./Images/Lakeflow_Designer_DAG.png "Lakeflow Designer DAG")

In [ ]:
%sql
-- Verify your Designer pipeline wrote the expected table.
-- Run this AFTER the pipeline finishes its first run.

SELECT
  team,
  season_points,
  points_finishes,
  ROUND(season_points * 1.0 / points_finishes, 2) AS avg_points_per_finish
FROM main.default.f1_team_points_finishes
ORDER BY season_points DESC;

## 🎯 Key takeaways

* You just built and scheduled a real ETL pipeline without writing any pipeline code.
* The output is a **standard Delta table** — every other tool in Databricks (SQL editor, dashboards, Genie, Power BI/Tableau via Lakeflow Connect) sees it like any other table.
* Unity Catalog permissions are **inherited automatically** — whoever can read the silver tables can read this one, no new grants needed.
* If you want this pipeline as code later, the engine underneath is the same as a SQL/Python Lakeflow pipeline — you can graduate to code when you outgrow the visual editor.

**Next:** `07_Dashboard_Builder.ipynb` — use Genie Code to spin up the workshop dashboard.